---
title: PLACEHOLDER
subtitle: Variant Summary
date: "9999-12-31"
edit_url: null
---



In [20]:
import altair as alt
from harpy.report.components import colored_boxes, print_html, standard_itable
from harpy.report.utilities import binned_histogram, nxx, last_line
from harpy.report.theme import palette
from io import StringIO
import itables
import os
import numpy as np
import pandas as pd
import sys
itables.init_notebook_mode(connected=True)

In [ ]:
infile = "variants.raw.stats"
vcf = "variants.raw"

In [ ]:
def extract_metric(x: list[str], param: str):
    '''Convenience function to find the relevnant sections of the bcftools.stats file and return a table'''
    input = "".join(s for s in x if s.startswith(param))
    return pd.read_table(StringIO(input), sep = "\t", header = None)

In [21]:
infile  = '/home/pdimens/Documents/harpy/SNP/mpileup/reports/data/variants.raw.stats'

In [22]:
dataL = open(infile, 'r').readlines()
bcf = infile.replace(".stats", ".bcf")

In [42]:
sn = extract_metric(dataL, "SN").drop(columns = [0,1])
sn.columns = ["Metric", "Number"]
sn['Metric'] = [i.replace("number of", '').replace(":",'') for i in sn['Metric']]
_ = sn.to_dict('list')
summ_stats = dict(zip(*(_.values())))
boxes = colored_boxes()
for k,v in summ_stats.items():
    _k = k.replace("SNP sites", "SNPs")
    boxes.add(v, _k)
boxes.render()

## SNPs

## Quality Statistics

These are per-locus SNP quality statistics, which correspond to the Quality (`QUAL`) values of `bcftools stats`. 
Each quality score is has a semi-open bin width of 10, where `0` would be considered "a QUAL score
greater than or equal to `0` and less than `10`". The histogram is truncated at the 99th percentile
for visual clarity.

In [ ]:
qual = extract_metric(dataL, "QUAL")[[2,3,6]]
qual.columns = ["Quality Score", "SNPs", "Indels"]
do_qual = True
if len(qual) == 0:
    do_qual = False
    print_html(f"Skipping - {os.path.basename(infile)} does not contain a QUAL section")
else:
    q99_snp = int(qual.loc[qual.index.repeat(qual['SNPs']), 'Quality Score'].quantile(0.99))
    qual['Quality_binned'] = (np.floor(qual['Quality Score'] / 10) * 10).astype(int)
    # Group by the new bins and sum SNPs and Indels
    rebinned = qual.groupby('Quality_binned').agg({
        'SNPs': 'sum',
        'Indels': 'sum'
    }).reset_index()
    rebinned['SNPs'] = rebinned['SNPs'] / rebinned['SNPs'].sum()
    rebinned['Indels'] = rebinned['Indels'] / rebinned['Indels'].sum()
    # Rename column back to Quality
    rebinned = rebinned.rename(columns={'Quality_binned': 'Quality Score'})
    (
        alt.Chart(rebinned.drop(columns = 'Indels'))
        .transform_filter(alt.FieldRangePredicate(field="Quality Score", range=[0, q99_snp]))
        .mark_area(interpolate = "monotone")
        .encode(
            x = alt.X('Quality Score:Q', title = "Genotype Quality Score", bin = 'binned').axis(tickMinStep = 10),
            y = alt.Y('SNPs:Q', title = "Percent of SNPs")
                .axis(format ='%'),
            tooltip = ['Quality Score:Q', alt.Tooltip("SNPs:Q", format = ".2%", bin = "binned")]
        )
        .properties(
        title=alt.Title('SNP Genotype Quality', subtitle = f'Scores are binned by 10 and truncated at 99th percentile ({q99_snp})'),
        usermeta={'embedOptions': {'downloadFileName': f'snp.quality'}}
    )
    ).display()

:::

### Depth per Genotype
These are per-locus statistics, which correspond to the Depth (`DP`) calculations of `bcftools stats`

In [170]:
dp = extract_metric(dataL, "DP").drop(columns = [0,1])
dp.columns = ["Bin", "Genotypes", "PercentGenotypes", "NumberSites", "PercentSites"]

if len(dp) == 0:
    print_html(f"Skipping - {os.path.basename(infile)} does not contain a DP section")
else:
    q99 = dp.loc[dp.index.repeat(dp['Genotypes']), 'Bin'].quantile(0.999)
    dp['PercentGenotypes'] = dp['PercentGenotypes']/ 100
    (
        alt.Chart(dp)
        .transform_filter(alt.FieldRangePredicate(field="Bin", range=[0, q99]))
        .mark_area(interpolate = "monotone")
        .encode(
            x = alt.X('Bin:Q', title = "Depth Bin", bin = 'binned').axis(tickMinStep = 5),
            y = alt.Y('PercentGenotypes:Q', title = "Percent of SNPs")
                .axis(format = "%"),
            tooltip = [
                alt.Tooltip('Bin:Q', title = 'Depth Bin'),
                alt.Tooltip("PercentGenotypes:Q", format = ".2%", bin = "binned")
            ]
        )
        .properties(
        title=alt.Title('Depth Per Genotype', subtitle = f'Bins are truncated at Q99 ({q99})'),
        usermeta={'embedOptions': {'downloadFileName': f'snp.quality'}}
    )
    ).display()

alt.Chart(...)

## Indels
### Indel Lengths
This is the distribution of insertions and deletions based on length and frequency. The deletion
lengths are presented as negative values for visual clarity.

In [158]:
idd = extract_metric(dataL, "IDD")[[2,3]]
idd.columns = ['Length', 'Count']
if len(idd) == 0:
    print_html(f"Skipping - {os.path.basename(infile)} does not contain a IDD section")
else:
    idd['Type'] = ["Insertion" if i > 0 else "Deletion" for i in idd['Length']]

    # Add Percent column (proportion of total count)
    idd['Ratio'] = (idd['Count'] / idd['Count'].sum()).round(4)

    # Add PercentType column (proportion within each Type)
    idd['RatioType'] = idd.groupby('Type')['Count'].transform(lambda x: x / x.sum()).round(4)
    (
        alt.Chart(idd)
        .mark_area(interpolate = "monotone")
        .encode(
            x = 'Length:Q',
            y = alt.Y('Ratio:Q', title = "Percent of Indels").axis(format = "%"),
            color = "Type:N",
            tooltip = [
                'Type:N',
                'Length:Q',
                alt.Tooltip('Ratio:Q', title = 'Percent (all)', format = ".2%"),
                alt.Tooltip('RatioType', title = 'Percent (type)', format = ".2%")
            ]
        )
        .configure_legend(orient = "top")
        .properties(
        title=alt.Title('Insertion-Deletion Distribution'),
        usermeta={'embedOptions': {'downloadFileName': f'indel.distribution'}}
    )
    ).display()

alt.Chart(...)

### Indel Genotype Quality
These are per-locus indel quality statistics, which correspond to the Quality (`QUAL`) values of `bcftools stats`. 
Each quality score is has a semi-open bin width of 10, where `0` would be considered "a QUAL score
greater than or equal to `0` and less than `10`". The histograms are truncated at the 99th percentile
for visual clarity.

In [174]:
if not do_qual:
    print_html(f"Skipping - {os.path.basename(infile)} does not contain a QUAL section")
else:
    q99_indel = int(qual.loc[qual.index.repeat(qual['Indels']), 'Quality Score'].quantile(0.99))
    (
    alt.Chart(rebinned.drop(columns = 'SNPs'))
    .transform_filter(alt.FieldRangePredicate(field="Quality Score", range=[0, q99_indel]))
    .mark_area(interpolate = "monotone")
    .encode(
        x = alt.X('Quality Score:Q', title = "Genotype Quality Score", bin = 'binned').axis(tickMinStep = 10),
        y = alt.Y('Indels:Q', title = "Percent of Indels")
            .axis(format ='%'),
        tooltip = ['Quality Score:Q', alt.Tooltip("Indels:Q", format = ".2%", bin = "binned")]
    )
    .properties(
    title=alt.Title('Indel Genotype Quality', subtitle = f'Scores are binned by 10 and truncated at 99th percentile ({q99_indel})'),
    usermeta={'embedOptions': {'downloadFileName': f'indel.quality'}}
)
).display()

alt.Chart(...)

:::

# Per-Sample
##

In [ ]:
.pscL <- grepl("^PSC", dataL)
psc <- read.table(text=dataL[.pscL])[ ,3:14]
names(psc) <- c("Sample", "HomRef", "HomAlt", "Het", "Transitions", "Transversions", "Indels",	"MeanDepth", "Singletons",	"HapRef", "HapAlt", "Missing")
tidy_psc <- pivot_longer(psc[,c(1,2,3,4,5,6,7,9,12)], -Sample , names_to = "Metric", values_to = "Count")
tidy_psc$Sample <- as.factor(basename(tidy_psc$Sample))
figheight <- 3 + (0.6 * nrow(psc))

::: {.card title="Individual Stats (Plot)"}
These are per-locus statistics, which correspond to the Per-Sample Counts (`PSC`) calculations of `bcftools stats`. The table reflects the data visualized in the plot tab. `HomAlt` and `HomRef` refer to homozygous for the Reference and Alternative alleles, respectively.

In [ ]:
highchart() |>
  hc_chart(inverted=TRUE, animation = F) |>
  hc_add_series(data = tidy_psc, type = "scatter", hcaes(y = Count, x = Sample, group = Metric)) |>
  hc_colors(c("#41739c", "#eb7e2b", "#aa042e", "#837fbd", "#202020", "#9dc96c", "#E6AB02", "#d39ab7")) |>
  hc_title(text = "Individual Statistics") |>
  hc_xAxis(type = "category", title = list(text = ""), categories = levels(tidy_psc$Sample)) |>
  hc_yAxis(title = list(text = "count")) |>
  hc_tooltip(
    crosshairs = TRUE, animation = F,
    pointFormat= '<b>{point.category}</b><br>Count: <b>{point.y:.0f}</b>'
  ) |>
  hc_exporting(enabled = T, filename = "samples.variantstats",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
  )

:::

In [ ]:
DT::datatable(
  psc, rownames = F,
  fillContainer = T,
  filter = "top",
  extensions = 'Buttons',
  options = list(
    dom = 'Brtp',
    buttons = list(list(extend = "csv",filename = "snp_indiv_stats")),
    scrollX = TRUE,
    paging = F
  )
)